# Ensemble Affinity Adjudication Analysis

This notebook compares benchmark model scores with final adjudication outputs in two ordered pipelines:

1. **RA analysis** runs first from the RA merged and adjudicated files.
2. **PRP analysis** runs second only when both PRP input files are present.

Figures are shown inline and no summary CSVs are created.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import plotly.express as px

In [ ]:
# Shared configuration and schema alignment
ignore_models = [
    'smollm2-1.7b',
    'gemma3-4b',
    'phi3-3.8b',
    'deepseek-v3.2-cloud',
    'qwen3.5-397b-cloud',
]
run_id = 'tpwrs_2026_tpwrs_2026'  # Set to None to use the latest run folder.

repo_root = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
bench_root = repo_root / 'data' / 'results' / 'affinity_benchmark'

if run_id is None:
    run_dirs = sorted(path for path in bench_root.iterdir() if path.is_dir())
    if not run_dirs:
        raise FileNotFoundError(f'No benchmark runs found under {bench_root}')
    run_dir = run_dirs[-1]
else:
    run_dir = bench_root / run_id

def append_final_rows(base_df, adjudication_df, cosine_column=None):
    column_renames = {
        'Final_Affinity': 'LLM_Affinity',
        'Final_Reason': 'LLM_Affinity_Reason',
    }
    if cosine_column is not None:
        column_renames['Cosine_x100_Mean'] = cosine_column

    final_rows = adjudication_df.copy().rename(columns=column_renames)
    provided_slug = final_rows['Model_Slug'].astype('string')
    final_method = final_rows['Final_Method'].fillna('adjudicator').astype(str)
    final_rows['Model_Slug'] = np.where(
        provided_slug.notna(),
        provided_slug.astype(str) + '_adjudicator',
        final_method,
    )

    for column in base_df.columns:
        if column not in final_rows.columns:
            final_rows[column] = pd.NA

    return pd.concat([base_df, final_rows[base_df.columns]], ignore_index=True)

print('Run directory:', run_dir)

# 1. RA Analysis

Load and prepare RA benchmark and final rows, then analyze score distributions and the highest- and lowest-agreement abstract-question pairs.

In [ ]:
# RA input and processing
ra_path = run_dir / 'merged_ra_affinities.csv'
ra_adjudication_path = run_dir / 'adjudicated_ra_affinities.csv'

missing_ra_inputs = [
    path for path in (ra_path, ra_adjudication_path) if not path.exists()
]
if missing_ra_inputs:
    raise FileNotFoundError(
        'Missing required RA input(s): ' + ', '.join(str(path) for path in missing_ra_inputs)
)

ra_df = pd.read_csv(ra_path)
ra_df = ra_df[~ra_df['Model_Slug'].isin(ignore_models)].copy()
ra_benchmark_models = set(ra_df['Model_Slug'].dropna().astype(str))
ra_adjudication_df = pd.read_csv(ra_adjudication_path)
ra_df = append_final_rows(
    ra_df,
    ra_adjudication_df,
    cosine_column='Cosine_x100',
)

mapping_path = repo_root / 'data' / 'processed' / 'ra_grouping_ra2025.csv'
ra_grouping_df = pd.read_csv(mapping_path)
required_mapping_cols = {'RA2025', 'Grouping'}
missing_mapping_cols = required_mapping_cols - set(ra_grouping_df.columns)
if missing_mapping_cols:
    raise ValueError(f'Missing required mapping columns: {missing_mapping_cols}')

ra_to_prp_map = (
    ra_grouping_df[['RA2025', 'Grouping']]
    .dropna(subset=['RA2025', 'Grouping'])
    .drop_duplicates()
    .rename(columns={'RA2025': 'RA2025_ID', 'Grouping': 'PRP_Name'})
)
ra_to_prp_map['RA2025_ID'] = pd.to_numeric(
    ra_to_prp_map['RA2025_ID'],
    errors='coerce',
).astype('Int64')
ra_df['RA2025_ID'] = pd.to_numeric(ra_df['RA2025_ID'], errors='coerce').astype('Int64')
ra_df = ra_df.merge(ra_to_prp_map, on='RA2025_ID', how='left')

print('RA rows:', len(ra_df), '| Comparison series:', ra_df['Model_Slug'].nunique())
display(ra_df.head(3))

## RA Distributions and Agreement Cases

In [ ]:
fig = px.box(
    ra_df,
    x='RA2025_ID',
    y='LLM_Affinity',
    color='Model_Slug',
    title='RA Affinity Distribution by Question and Comparison Series',
)
fig.update_layout(xaxis_title='RA Question', yaxis_title='LLM Affinity')
fig.show()

top_ra_disagreement_ids = (
    ra_adjudication_df.loc[ra_adjudication_df['agreement_mean'] < 1]
    .nlargest(20, 'Affinity_Range')['Abstract_Index']
    .drop_duplicates()
    .head(5)
)
high_ra_agreement_ids = (
    ra_adjudication_df.loc[ra_adjudication_df['agreement_mean'] == 1]
    .nsmallest(20, 'Affinity_Range')['Abstract_Index']
    .drop_duplicates()
    .head(5)
)

for abstract_ids, title in [
    (top_ra_disagreement_ids, 'Highest-Disagreement RA Cases'),
    (high_ra_agreement_ids, 'Highest-Agreement RA Cases'),
]:
    fig = px.scatter(
        ra_df[ra_df['Abstract_Index'].isin(abstract_ids)],
        x='RA2025_ID',
        y='LLM_Affinity',
        color='Model_Slug',
        facet_col='Abstract_Index',
        hover_data=['Model_Slug', 'PRP_Name'],
        title=title,
    )
    fig.update_layout(xaxis_title='RA Question', yaxis_title='LLM Affinity')
    fig.show()

In [ ]:
selected_ra_abstract = top_ra_disagreement_ids.iloc[0]
base_ra_df = ra_df[
    (ra_df['Abstract_Index'] == selected_ra_abstract)
    & (ra_df['Model_Slug'].isin(ra_benchmark_models))
]
final_ra_df = ra_df[
    (ra_df['Abstract_Index'] == selected_ra_abstract)
    & (~ra_df['Model_Slug'].isin(ra_benchmark_models))
]

fig = px.box(
    base_ra_df,
    x='RA2025_ID',
    y='LLM_Affinity',
    points='all',
    color='PRP_Name',
    category_orders={
        'PRP_Name': ra_to_prp_map['PRP_Name'].drop_duplicates().tolist()
    },
    title=f'RA Affinity Distribution for {selected_ra_abstract}',
)
fig.add_scatter(
    x=final_ra_df['RA2025_ID'],
    y=final_ra_df['LLM_Affinity'],
    mode='markers',
    name='Final affinity',
    marker=dict(size=6, symbol='x', color='black'),
    hovertext=final_ra_df['PRP_Name'],
    hovertemplate='RA2025_ID=%{x}<br>LLM_Affinity=%{y}<br>PRP=%{hovertext}<extra></extra>',
)
fig.update_layout(
    xaxis_title='RA Question',
    yaxis_title='LLM Affinity',
    width=800,
    height=500,
    legend=dict(title=None, orientation='h', yanchor='bottom', y=-0.4, xanchor='right', x=1),
)
fig.show()

# 2. PRP Analysis

Load, prepare, and analyze PRP benchmark and final rows only when both required PRP files are available.

In [ ]:
# PRP input and processing
prp_path = run_dir / 'merged_prp_affinities.csv'
prp_adjudication_path = run_dir / 'adjudicated_prp_affinities.csv'
prp_available = prp_path.exists() and prp_adjudication_path.exists()

if prp_available:
    prp_df = pd.read_csv(prp_path)
    prp_df = prp_df[~prp_df['Model_Slug'].isin(ignore_models)].copy()
    prp_benchmark_models = set(prp_df['Model_Slug'].dropna().astype(str))
    prp_adjudication_df = pd.read_csv(prp_adjudication_path)
    prp_df = append_final_rows(prp_df, prp_adjudication_df)

    print('PRP rows:', len(prp_df), '| Comparison series:', prp_df['Model_Slug'].nunique())
    display(prp_df.head(3))
else:
    missing_prp_inputs = [
        str(path) for path in (prp_path, prp_adjudication_path) if not path.exists()
    ]
    print('PRP analysis skipped. Missing input(s):')
    for missing_path in missing_prp_inputs:
        print('-', missing_path)

## PRP Distributions and Agreement Cases

In [ ]:
if prp_available:
    fig = px.box(
        prp_df,
        x='PRP_Name',
        y='LLM_Affinity',
        color='Model_Slug',
        title='PRP Affinity Distribution by Programme and Comparison Series',
    )
    fig.update_layout(xaxis_title='PRP Name', yaxis_title='LLM Affinity')
    fig.show()

    top_prp_disagreement_ids = (
        prp_adjudication_df.loc[prp_adjudication_df['agreement_mean'] < 1]
        .nlargest(20, 'Affinity_Range')['Abstract_Index']
        .drop_duplicates()
        .head(5)
    )
    high_prp_agreement_ids = (
        prp_adjudication_df.loc[prp_adjudication_df['agreement_mean'] == 1]
        .nsmallest(20, 'Affinity_Range')['Abstract_Index']
        .drop_duplicates()
        .head(5)
    )

    for abstract_ids, title in [
        (top_prp_disagreement_ids, 'Highest-Disagreement PRP Cases'),
        (high_prp_agreement_ids, 'Highest-Agreement PRP Cases'),
    ]:
        fig = px.scatter(
            prp_df[prp_df['Abstract_Index'].isin(abstract_ids)],
            x='PRP_Name',
            y='LLM_Affinity',
            color='Model_Slug',
            facet_col='Abstract_Index',
            title=title,
        )
        fig.update_layout(yaxis_title='LLM Affinity')
        fig.show()
else:
    print('PRP distribution and agreement plots skipped.')

In [ ]:
if prp_available:
    pd.set_option('display.max_colwidth', None)
    top_disagreement_prp = (
        prp_adjudication_df[
            prp_adjudication_df['Abstract_Index'].isin(top_prp_disagreement_ids)
        ]
        .sort_values(
            ['Abstract_Index', 'Affinity_Range', 'agreement_mean'],
            ascending=[True, False, True],
        )
        .groupby('Abstract_Index', as_index=False)
        .head(1)[
            [
                'Abstract_Index',
                'Document Title',
                'PRP_Name',
                'Final_Affinity',
                'Final_Reason',
                'Affinity_Range',
                'agreement_mean',
            ]
        ]
        .reset_index(drop=True)
    )
    display(top_disagreement_prp.rename(columns={'Final_Reason': 'Final Reason'}))

    for row in top_disagreement_prp.itertuples(index=False):
        print(f'Abstract {row.Abstract_Index} | PRP: {row.PRP_Name}')
        print(f'Final Reason: {row.Final_Reason if pd.notna(row.Final_Reason) else "N/A"}')
        print('-' * 120)

    selected_prp_abstract = top_prp_disagreement_ids.iloc[0]
    base_prp_df = prp_df[
        (prp_df['Abstract_Index'] == selected_prp_abstract)
        & (prp_df['Model_Slug'].isin(prp_benchmark_models))
    ]
    final_prp_df = prp_df[
        (prp_df['Abstract_Index'] == selected_prp_abstract)
        & (~prp_df['Model_Slug'].isin(prp_benchmark_models))
    ]

    fig = px.box(
        base_prp_df,
        x='PRP_Name',
        y='LLM_Affinity',
        points='all',
        color='PRP_Name',
        title=f'PRP Affinity Distribution for {selected_prp_abstract}',
    )
    fig.add_scatter(
        x=final_prp_df['PRP_Name'],
        y=final_prp_df['LLM_Affinity'],
        mode='markers',
        name='Final affinity',
        marker=dict(size=6, symbol='x', color='black'),
    )
    fig.update_layout(
        xaxis_title='PRP Name',
        yaxis_title='LLM Affinity',
        width=700,
        height=500,
    )
    fig.show()
else:
    print('PRP reason inspection and selected-case plot skipped.')